In [5]:
# ==============================
# HARD VOTING ENSEMBLE (with remapping)
# Combines: CNN, BiGRU, DistilBERT, BERT — all aligned to canonical class order
# ==============================
import pandas as pd
import numpy as np
import re
import os
import pickle
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from nltk.stem import WordNetLemmatizer
import torch
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    BertTokenizerFast,
    BertForSequenceClassification
)
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from bs4 import BeautifulSoup
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk import pos_tag
from collections import defaultdict
import warnings
from scipy.optimize import linear_sum_assignment
warnings.filterwarnings('ignore')

# === NLTK setup (quiet) ===
import nltk
nltk.download('punkt', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)

# === Reproducibility ===
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# === Canonical class order — MUST match training ===
y_dict = {
    'self direction': 0, 'stimulation': 1, 'hedonism': 2, 'achievement': 3, 'power': 4,
    'security': 5, 'conformity': 6, 'tradition': 7, 'benevolence': 8, 'universalism': 9
}
class_labels = [k for k in y_dict.keys()]
n_classes = len(class_labels)
idx_to_class = {v: k for k, v in y_dict.items()}

print("✅ Canonical class order (0→9):")
for i, name in enumerate(class_labels):
    print(f"{i}: {name}")

# === Load & Prepare Data (identical to soft voting) ===
df = pd.read_csv('newextendeddataset.csv', encoding='utf-8-sig')
df['category_clean'] = df['category'].str.strip().str.lower()
df['label_id'] = df['category_clean'].map(y_dict)
df = df.dropna(subset=['label_id'])
df['label_id'] = df['label_id'].astype(int)

texts = df['Base_Reviews'].values
y_all = df['label_id'].values

# 🔑 Exact same split as soft voting script
X_train_raw, X_test_raw, y_train_raw, y_test_true = train_test_split(
    texts, y_all,
    test_size=0.15,
    random_state=SEED,
    stratify=y_all
)

print(f"\n✅ Final Splits (aligned with all models):")
print(f"   Train (85%): {len(X_train_raw)} samples")
print(f"   Test  (15%): {len(X_test_raw)} samples ← frozen")

# === Preprocessing (Exact replica of RNN training script) ===
tag_map = defaultdict(lambda: 'n')
tag_map['J'] = 'a'  # adjective
tag_map['V'] = 'v'  # verb
tag_map['R'] = 'r'  # adverb

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text_for_rnn(text):
    if not isinstance(text, str):
        return ""
    # 1. HTML stripping
    soup = BeautifulSoup(text, "html.parser")
    text = soup.get_text()
    # 2. URL removal
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 3. Cleaning
    text = text.lower()
    text = re.sub(r'[/(){}\[\]\<\>\|@,;]', ' ', text)
    text = re.sub(r'[^a-z #+_]', '', text)
    # 4. Tokenize + POS-aware lemmatize + stopword removal
    tokens = word_tokenize(text)
    final_words = []
    for word, pos in pos_tag(tokens):
        if word not in stop_words and word.isalpha():
            pos_tag_simple = tag_map[pos[0]] if pos else 'n'
            lemma = lemmatizer.lemmatize(word, pos=pos_tag_simple)
            final_words.append(lemma)
    return ' '.join(final_words)

# === Device Setup ===
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

if device.type == "mps":
    os.environ['PYTORCH_MPS_HIGH_WATERMARK_RATIO'] = '0.0'

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

import tensorflow as tf
tf.config.set_visible_devices([], 'GPU')
print("✅ TensorFlow will use CPU only")

# === Load Models ===
print("\nLoading models...")

# CNN
cnn_model = load_model('./final_cnn_model.h5')
with open('./final_cnn_tokenizer.pkl', 'rb') as f:
    cnn_tokenizer = pickle.load(f)
print("✅ CNN loaded")

# BiGRU
bigru_model = load_model('./final_bigru_model.h5')
with open('./final_bigru_tokenizer.pkl', 'rb') as f:
    bigru_tokenizer = pickle.load(f)
print("✅ BiGRU loaded")

# DistilBERT
distilbert_model = DistilBertForSequenceClassification.from_pretrained('./distilbert_best_cv')
distilbert_tokenizer = DistilBertTokenizerFast.from_pretrained('./distilbert_best_cv')
distilbert_model.to(device).eval()
print("✅ DistilBERT loaded")

# BERT
bert_path = './bert_best_cv'
bert_model = BertForSequenceClassification.from_pretrained(bert_path)
bert_tokenizer = BertTokenizerFast.from_pretrained(bert_path)
bert_model.to(device).eval()
print("✅ BERT loaded")

# === 🔑 STEP: Infer REMAPPINGS (same as soft voting) ===
print("\n" + "="*60)
print("🔍 INFERRING CLASS REMAPPING FOR CNN & BiGRU (Hungarian)")
print("="*60)

def batched_predict_proba(predict_func, texts, batch_size=16):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        probs = predict_func(batch)
        all_probs.append(probs)
    return np.vstack(all_probs)

def get_cnn_raw_proba(texts):
    cleaned = [clean_text_for_rnn(t) for t in texts]
    seqs = cnn_tokenizer.texts_to_sequences(cleaned)
    padded = pad_sequences(seqs, maxlen=100, padding='post')
    return cnn_model.predict(padded, verbose=0)

def get_bigru_raw_proba(texts):
    cleaned = [clean_text_for_rnn(t) for t in texts]
    seqs = bigru_tokenizer.texts_to_sequences(cleaned)
    padded = pad_sequences(seqs, maxlen=100, padding='post')
    return bigru_model.predict(padded, verbose=0)

def infer_optimal_remapping(raw_predict_func, texts, y_true, name):
    print(f"\n→ Inferring optimal {name} remapping (Hungarian)...")
    probs = batched_predict_proba(raw_predict_func, texts)
    y_pred = np.argmax(probs, axis=1)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for i in range(len(y_pred)):
        cm[y_pred[i], y_true[i]] += 1
    row_ind, col_ind = linear_sum_assignment(-cm)
    remap = col_ind.tolist()
    for i in range(n_classes):
        best_true = remap[i]
        count = cm[i, best_true]
        print(f"  {name} output[{i:2}] → true class {best_true:2} ({class_labels[best_true]:15}) | matches: {count}")
    return remap

# Use full test set for stable mapping
CNN_REMAP = infer_optimal_remapping(get_cnn_raw_proba, X_test_raw, y_test_true, "CNN")
BIGRU_REMAP = infer_optimal_remapping(get_bigru_raw_proba, X_test_raw, y_test_true, "BiGRU")

print(f"\n✅ Final remappings:")
print(f"CNN_REMAP    = {CNN_REMAP}")
print(f"BIGRU_REMAP  = {BIGRU_REMAP}")

# === ✅ REMAPPING HELPER ===
def remap_predictions(preds, mapping):
    """Map raw model predictions (0–9) → canonical class indices using mapping."""
    remapped = np.array([mapping[p] for p in preds])
    return remapped

# === Predict Functions — with REMAPPING for CNN & BiGRU ===
def predict_cnn(texts):
    probs = get_cnn_raw_proba(texts)
    raw_preds = np.argmax(probs, axis=1)
    return remap_predictions(raw_preds, CNN_REMAP)

def predict_bigru(texts):
    probs = get_bigru_raw_proba(texts)
    raw_preds = np.argmax(probs, axis=1)
    return remap_predictions(raw_preds, BIGRU_REMAP)

def predict_distilbert(texts):
    inputs = distilbert_tokenizer(
        [str(t) for t in texts],
        padding=True, truncation=True, max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = distilbert_model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
    return preds  # assumed canonical — no remap needed

def predict_bert(texts):
    inputs = bert_tokenizer(
        [str(t) for t in texts],
        padding=True, truncation=True, max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = bert_model(**inputs).logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
    return preds  # assumed canonical

# === Batched Prediction Wrapper (for efficiency) ===
def batched_predict(predict_func, texts, batch_size=16):
    all_preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        preds = predict_func(batch)
        all_preds.append(preds)
    return np.concatenate(all_preds)

# === Hard Voting Ensemble (NOW WITH REMAPPING!) ===
def ensemble_hard_voting(texts, batch_size=16):
    pred_cnn = batched_predict(predict_cnn, texts, batch_size)
    pred_bigru = batched_predict(predict_bigru, texts, batch_size)
    pred_distil = batched_predict(predict_distilbert, texts, batch_size)
    pred_bert = batched_predict(predict_bert, texts, batch_size)

    # Stack: shape (n_samples, 4)
    all_preds = np.vstack([pred_cnn, pred_bigru, pred_distil, pred_bert]).T

    # Majority vote (break ties by picking most frequent; if tie, pick smallest index)
    final_preds = []
    for row in all_preds:
        vote_counts = Counter(row)
        max_count = vote_counts.most_common(1)[0][1]
        # Get all classes with max count
        top_classes = [cls for cls, cnt in vote_counts.items() if cnt == max_count]
        # Break tie: choose class with highest *average probability* (optional)
        # Here: simpler — choose smallest class index (deterministic)
        final_vote = min(top_classes)
        final_preds.append(final_vote)
    return np.array(final_preds)

# === Run Hard Voting ===
print("\n" + "="*60)
print("🚀 RUNNING HARD VOTING ENSEMBLE (with remapping)")
print("="*60)

y_pred_ensemble = ensemble_hard_voting(X_test_raw, batch_size=16)

# === Evaluation with clean metrics ===
from sklearn.metrics import accuracy_score, f1_score

acc = accuracy_score(y_test_true, y_pred_ensemble)
macro_f1 = f1_score(y_test_true, y_pred_ensemble, average='macro')
weighted_f1 = f1_score(y_test_true, y_pred_ensemble, average='weighted')

print(f"\n✅ Hard Voting Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"✅ Hard Voting Macro F1:      {macro_f1:.4f} ({macro_f1*100:.2f}%)")
print(f"✅ Hard Voting Weighted F1:   {weighted_f1:.4f} ({weighted_f1*100:.2f}%)")

print("\n=== HARD VOTING CLASSIFICATION REPORT (with remapping) ===")
print(classification_report(y_test_true, y_pred_ensemble, target_names=class_labels, zero_division=0, digits=4))

# Optional: Save
# np.save('hard_voting_predictions.npy', y_pred_ensemble)
# print("✅ Predictions saved.")

✅ Canonical class order (0→9):
0: self direction
1: stimulation
2: hedonism
3: achievement
4: power
5: security
6: conformity
7: tradition
8: benevolence
9: universalism

✅ Final Splits (aligned with all models):
   Train (85%): 9469 samples
   Test  (15%): 1671 samples ← frozen

Using device: mps
✅ TensorFlow will use CPU only

Loading models...


✅ CNN loaded
✅ BiGRU loaded
✅ DistilBERT loaded
✅ BERT loaded

🔍 INFERRING CLASS REMAPPING FOR CNN & BiGRU (Hungarian)

→ Inferring optimal CNN remapping (Hungarian)...
  CNN output[ 0] → true class  0 (self direction ) | matches: 351
  CNN output[ 1] → true class  1 (stimulation    ) | matches: 106
  CNN output[ 2] → true class  2 (hedonism       ) | matches: 190
  CNN output[ 3] → true class  3 (achievement    ) | matches: 49
  CNN output[ 4] → true class  4 (power          ) | matches: 35
  CNN output[ 5] → true class  5 (security       ) | matches: 70
  CNN output[ 6] → true class  6 (conformity     ) | matches: 157
  CNN output[ 7] → true class  7 (tradition      ) | matches: 5
  CNN output[ 8] → true class  8 (benevolence    ) | matches: 134
  CNN output[ 9] → true class  9 (universalism   ) | matches: 4

→ Inferring optimal BiGRU remapping (Hungarian)...
  BiGRU output[ 0] → true class  0 (self direction ) | matches: 348
  BiGRU output[ 1] → true class  1 (stimulation    ) | mat